In [0]:
-- Count rows in ad_click_logs
-- Expected: ~505,000 rows
SELECT COUNT(*) as ad_click_logs_count
FROM adtech_catalog.bronze.ad_click_logs;

ad_click_logs_count
2505000


In [0]:
-- Count rows in ad_metadata_catalog
-- Expected: ~1,050 rows
SELECT COUNT(*) as ad_metadata_catalog_count
FROM adtech_catalog.bronze.ad_metadata_catalog;

ad_metadata_catalog_count
10500


In [0]:
-- Summary of both tables
SELECT 
    'ad_click_logs' as table_name,
    COUNT(*) as row_count
FROM adtech_catalog.bronze.ad_click_logs
UNION ALL
SELECT 
    'ad_metadata_catalog' as table_name,
    COUNT(*) as row_count
FROM adtech_catalog.bronze.ad_metadata_catalog;

table_name,row_count
ad_click_logs,2505000
ad_metadata_catalog,10500


In [0]:
-- Sample data from ad_click_logs (check for anomalies)
SELECT * 
FROM adtech_catalog.bronze.ad_click_logs 
LIMIT 5;

User_ID,Click_Timestamp,Ad_Reference_ID,Ad_Type,Watch_Duration,user_age,device,platform_source,user_gender,user_clicked,ingestion_timestamp,ingestion_date,source_file,ingestion_batch_id,environment,git_commit,processing_status
f1bb8fa5-c70e-4bdf-a563-09d1c3828981,30/03/2026 18:18,AD_451788,Image,999999.0,125,Desktop,facebok,Other,0,2026-08-02T10:11:18.245Z,2026-08-02,raw_synthetic_ad_click_data.csv,20260802_101104,development,local,RAW
f4d9ddca-328b-461d-a7f4-ed0df77bb295,2026-02-28 05:35:33,AD_916180,null,0.0,20,null,google,Other,0,2026-08-02T10:11:18.245Z,2026-08-02,raw_synthetic_ad_click_data.csv,20260802_101104,development,local,RAW
89051d1a-2996-40c2-9bce-abe6f7200f99,2026-03-21 21:00:05,AD_970956,null,9.9,63,null,instagram,Female,0,2026-08-02T10:11:18.245Z,2026-08-02,raw_synthetic_ad_click_data.csv,20260802_101104,development,local,RAW
2a3bc74f-e488-4f0a-b1a2-453c3bbb6d5f,2026-01-26 03:41:22,CLK_762805,null,3.6,29,Tablet,instagram,Male,0,2026-08-02T10:11:18.245Z,2026-08-02,raw_synthetic_ad_click_data.csv,20260802_101104,development,local,RAW
7fb55cfd-c078-4fe7-a1c8-33c4ae179653,2026-02-23 02:02:06,AD_316809,Image,0.0,47,Tablet,facebook,Female,0,2026-08-02T10:11:18.245Z,2026-08-02,raw_synthetic_ad_click_data.csv,20260802_101104,development,local,RAW


In [0]:
-- View all columns in anomaly_report
DESCRIBE adtech_catalog.monitoring.anomaly_report;

col_name,data_type,comment
table,string,null
anomaly_type,string,null
count,bigint,null
detection_timestamp,timestamp,null
detection_date,date,null
batch_id,string,null
environment,string,null
git_commit,string,null
status,string,null
# Partition Information,,


In [0]:
-- All anomalies with their counts (sorted by count)
-- Shows what data quality issues were found
SELECT 
    table,
    anomaly_type,
    count,
    detection_timestamp
FROM adtech_catalog.monitoring.anomaly_report
ORDER BY count DESC;

table,anomaly_type,count,detection_timestamp
User Events,Platform - Unknown,418287,2026-08-02T10:24:12.369Z
User Events,Device - Typo moble,417774,2026-08-02T10:24:12.369Z
User Events,Device - Uppercase DESKTOP,417752,2026-08-02T10:24:12.369Z
User Events,Ad_Type - Lowercase video,417634,2026-08-02T10:24:12.369Z
User Events,Device - Null,417547,2026-08-02T10:24:12.369Z
User Events,Ad_Type - Uppercase IMAGE,417519,2026-08-02T10:24:12.369Z
User Events,Platform - Typo facebok,417266,2026-08-02T10:24:12.369Z
User Events,Platform - Typo gogle,417198,2026-08-02T10:24:12.369Z
User Events,Ad_Type - Null,416256,2026-08-02T10:24:12.369Z
Ad Catalog,Total Rows,10500,2026-08-02T10:24:12.369Z


In [0]:
-- Summary of anomalies by pipeline run
-- Shows total anomalies found in each run
SELECT 
    batch_id,
    MAX(detection_timestamp) as detection_time,
    SUM(count) as total_anomalies
FROM adtech_catalog.monitoring.anomaly_report
GROUP BY batch_id
ORDER BY detection_time DESC;

batch_id,detection_time,total_anomalies
20260802_102336,2026-08-02T10:24:12.369Z,3784156


In [0]:
-- Critical anomalies only (need immediate attention)
SELECT 
    anomaly_type,
    count,
    detection_timestamp
FROM adtech_catalog.monitoring.anomaly_report
WHERE anomaly_type LIKE '%CRITICAL%' 
   OR anomaly_type LIKE '%Corruption%'
   OR anomaly_type LIKE '%Logical Conflict%'
ORDER BY count DESC;

anomaly_type,count,detection_timestamp
Video Length - Logical Conflict,854,2026-08-02T10:24:12.369Z
Watch_Duration - Logical Conflict,213,2026-08-02T10:24:12.369Z
Device - Corruption CRITICAL,0,2026-08-02T10:24:12.369Z


In [0]:
-- View complete version history
SELECT * 
FROM adtech_catalog.monitoring.version_history 
ORDER BY deployed_at DESC;

version_id,environment,git_commit,deployed_at,description,status
20260802_110305,development,local,2026-08-02T11:03:49.762384,Export Gold to S3,SUCCESS
20260802_105134,development,local,2026-08-02T10:52:16.451157,Gold Quality Review,SUCCESS
20260802_104147,development,local,2026-08-02T10:42:36.403009,Gold Layer - Feature Engineering,SUCCESS
20260802_103154,development,local,2026-08-02T10:32:40.231671,Silver Layer - Data Cleaning,SUCCESS
20260802_102336,development,local,2026-08-02T10:24:15.900664,Anomaly Detection - Bronze Layer,SUCCESS
20260802_101104,development,local,2026-08-02T10:11:49.831122,Bronze Layer - Initial Load,SUCCESS


In [0]:
-- Show only anomaly detection runs
SELECT 
    version_id,
    deployed_at,
    description,
    status
FROM adtech_catalog.monitoring.version_history
WHERE description LIKE '%Anomaly%'
ORDER BY deployed_at DESC;

version_id,deployed_at,description,status
20260802_102336,2026-08-02T10:24:15.900664,Anomaly Detection - Bronze Layer,SUCCESS


In [0]:
-- Show only bronze load runs
SELECT 
    version_id,
    deployed_at,
    description,
    status
FROM adtech_catalog.monitoring.version_history
WHERE description LIKE '%Bronze%'
ORDER BY deployed_at DESC;

version_id,deployed_at,description,status
20260802_102336,2026-08-02T10:24:15.900664,Anomaly Detection - Bronze Layer,SUCCESS
20260802_101104,2026-08-02T10:11:49.831122,Bronze Layer - Initial Load,SUCCESS


In [0]:
-- Pipeline health summary
SELECT 
    'Bronze Layer' as layer,
    CASE 
        WHEN COUNT(*) > 0 THEN ' EXISTS'
        ELSE ' MISSING'
    END as status,
    COUNT(*) as row_count
FROM adtech_catalog.bronze.ad_click_logs
UNION ALL
SELECT 
    'Anomaly Report' as layer,
    CASE 
        WHEN COUNT(*) > 0 THEN ' EXISTS'
        ELSE ' MISSING'
    END as status,
    COUNT(*) as row_count
FROM adtech_catalog.monitoring.anomaly_report
UNION ALL
SELECT 
    'Version History' as layer,
    CASE 
        WHEN COUNT(*) > 0 THEN ' EXISTS'
        ELSE ' MISSING'
    END as status,
    COUNT(*) as row_count
FROM adtech_catalog.monitoring.version_history;


layer,status,row_count
Bronze Layer,EXISTS,2505000
Anomaly Report,EXISTS,42
Version History,EXISTS,6
